<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/03_baseline_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Baselines

DI725 Term Project — Multimodal Fusion for Remote Sensing Land Cover Segmentation.

This notebook trains and evaluates the vision-only baselines used as reference points for the multimodal experiments in `04_multimodal.ipynb`:

1. UNetFormer (no augmentation) — reproduces the PoC baseline.
2. UNetFormer (with augmentation) — Phase 2 baseline.
3. SegFormer (with augmentation) — second Phase 2 baseline.

All models share the same train/val/test split (from `02_preprocessing.ipynb`), pre-converted class-index masks, weighted cross-entropy loss with median frequency balancing, and 30-epoch training schedule.

Sections:
1. Setup
2. Dataset and DataLoaders
3. UNetFormer architecture
4. UNetFormer baseline (no augmentation)
5. UNetFormer baseline (with augmentation)
6. SegFormer baseline (with augmentation)
7. Test set evaluation
8. Save results

## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies not present in Colab by default
!pip install transformers timm einops -q

In [3]:
# Project paths (Drive) and a local working copy for fast I/O during training
from pathlib import Path
import shutil, os

PROJECT_DIR     = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE  = PROJECT_DIR / 'data'
SPLITS_CSV      = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints'
RESULTS_DIR     = PROJECT_DIR / 'results'
CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Local SSD copy avoids Drive I/O bottleneck during training
LOCAL_DATA  = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

print(f'Images: {LOCAL_IMAGES}')
print(f'Masks : {LOCAL_MASKS_CLASS}')

Copying data to local SSD...
Done.
Images: /content/data/images
Masks : /content/data/masks_class


In [4]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


In [5]:
# Class definitions
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

In [6]:
# Load the split CSV from Drive
split_df = pd.read_csv(SPLITS_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

Train: 7000  |  Val: 1500  |  Test: 1500


In [7]:
# Class weights via median frequency balancing, computed from the training split.
# Used for weighted cross-entropy loss in all three baselines.
class_avg = train_df[CLASS_NAMES].mean().values
class_freq = class_avg / class_avg.sum()
median_freq = np.median(class_freq)
class_weights = median_freq / (class_freq + 1e-8)

print(f"{'Class':<12} {'Avg %':>8} {'Frequency':>10} {'Weight':>8}")
print('-' * 42)
for i, c in enumerate(CLASS_NAMES):
    print(f'{c:<12} {class_avg[i]:>7.1f}% {class_freq[i]:>10.4f} {class_weights[i]:>8.2f}')

class_weights_tensor = torch.FloatTensor(class_weights).to(device)

Class           Avg %  Frequency   Weight
------------------------------------------
Tree            28.7%     0.2873     0.15
Shrub            0.8%     0.0083     5.12
Grass           45.3%     0.4533     0.09
Crop            18.0%     0.1803     0.24
Built-up         1.1%     0.0113     3.77
Barren           4.3%     0.0426     1.00
Water            1.7%     0.0168     2.54


## 2. Dataset and DataLoaders

The dataset reads pre-converted class-index masks from `data/masks_class/` (produced in `02_preprocessing.ipynb`), so no per-batch RGB conversion is needed.

Augmentations applied jointly to image and mask:
- Random horizontal flip (p=0.5)
- Random vertical flip (p=0.5)
- Random 90° rotation (one of {0°, 90°, 180°, 270°})

Validation and test sets are not augmented. Images are normalized with ImageNet statistics (the backbones expect this).

In [8]:
# Dataset class — reads pre-converted class-index masks directly
class RSDataset(Dataset):
    def __init__(self, dataframe, images_dir, masks_dir, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.augment = augment
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def _augment(self, img, mask):
        # img: tensor (3, H, W), mask: tensor (H, W). Apply same transform to both.
        if torch.rand(1).item() < 0.5:
            img  = torch.flip(img,  dims=[2])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            img  = torch.flip(img,  dims=[1])
            mask = torch.flip(mask, dims=[0])
        k = torch.randint(0, 4, (1,)).item()  # 0, 90, 180, 270 degrees
        if k > 0:
            img  = torch.rot90(img,  k, dims=[1, 2])
            mask = torch.rot90(mask, k, dims=[0, 1])
        return img, mask

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['filename']
        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)  # (3, H, W) in [0, 1]
        mask = np.array(Image.open(self.masks_dir / fname))  # (H, W) uint8 with values 0..6
        mask = torch.from_numpy(mask).long()

        if self.augment:
            img, mask = self._augment(img, mask)

        img = self.normalize(img)
        return img, mask

In [11]:
# DataLoader config. Train datasets are recreated per section to toggle augmentation.
BATCH_SIZE  = 16
NUM_WORKERS = 4

# Val and test never get augmented, so we create them once
val_dataset   = RSDataset(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, augment=False)
test_dataset  = RSDataset(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, augment=False)

val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Val:  {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'Test: {len(test_dataset)} samples, {len(test_loader)} batches')

Val:  1500 samples, 94 batches
Test: 1500 samples, 94 batches
